In [65]:
import pandas as pd

data = pd.read_excel('menu_orders.xls')
def ct(x): return pd.Series(1, index=x[pd.notnull(x)])
data = pd.DataFrame(map(ct, data.values)).fillna(0)
print(data)

     b    d    c    a    e
0  1.0  1.0  0.0  0.0  0.0
1  1.0  0.0  1.0  0.0  0.0
2  1.0  1.0  1.0  1.0  0.0
3  1.0  0.0  0.0  1.0  0.0
4  1.0  0.0  1.0  0.0  0.0
5  1.0  0.0  0.0  1.0  0.0
6  1.0  0.0  1.0  1.0  1.0
7  1.0  0.0  1.0  1.0  0.0
8  0.0  0.0  1.0  1.0  1.0


In [66]:
def is_exist(full_list, subset):  # 判断项集是否存在
    for x in full_list:
        if x == subset:
            return True
    return False

In [67]:
def compute_support(Item):  # 计算支持度
    d = data.copy()
    for x in Item:
        d = d[d[x] > 0]
    return len(d) / len(data)

In [68]:
def compute_confidence(Item):  # 计算置信度
    if compute_support(Item[:-1]) == 0:
        return 0
    return compute_support(Item) / compute_support(Item[:-1])

In [69]:
def Apriori(data, support = 0.2, confidence = 0.5):
    support_series = data.sum()/len(data) # 支持度序列
    columns = [set(x) for x in list(support_series[support_series >= support].index)] # 筛选符合支持度的项
    results = pd.DataFrame(columns = [['support', 'confidence']])

    while len(columns) > 0:
        print(', '.join([str(x) for x in columns]))
        k_columns = list()
        for i in range(len(columns) - 1):
            for j in range(i + 1, len(columns)):
                Itemset = columns[i] | columns[j]
                if len(columns[i] ^ columns[j]) == 2 and not is_exist(k_columns, Itemset) and compute_support(Itemset) >= support:
                    k_columns.append(Itemset)

        columns = k_columns
        for x in columns:
            for y in x:
                Item = list(x - {y}) + [y]
                if compute_confidence(Item) >= confidence:
                    results.loc[''.join(sorted(x -{y})) + ' -> ' + y] = [compute_support(Item), compute_confidence(Item)]
        
    return results

In [71]:
Apriori(data, 0.5, 0.5)

{'b'}, {'c'}, {'a'}
{'b', 'c'}, {'b', 'a'}


,support,confidence
c -> b,0.555556,0.833333
b -> c,0.555556,0.625000
a -> b,0.555556,0.833333
b -> a,0.555556,0.625000
